## Pivot language for back-translation (English -> German -> English)

We use **German** as the pivot language for back-translation augmentation. Rationale:

- German has the highest-quality publicly available open-source MT system for the
  `en-de` direction (Helsinki-NLP / OPUS-MT and Facebook M2M-100 both report top BLEU
  on en-de among Indo-European pivots), which keeps back-translation noise low enough
  to preserve label-bearing content while still producing surface variation.
- German is morphologically richer than English (case marking, separable verbs,
  compounding), so the round-trip rephrasing tends to alter syntax more than a closer
  pivot (e.g. Dutch) while staying lexically faithful to the medical domain vocabulary.
- We did not pivot through Kinyarwanda or any African language: open-source en-rw MT
  quality is currently too low for safe round-trip augmentation in this domain.

**Caveat (added for the Limitations section).** The Rwandan half of the training corpus
was originally translated from Kinyarwanda into English before being fed to the
classifiers. Back-translation augmentation on those rows therefore produces
**triple-translated examples** (rw -> en -> de -> en). We treat this as a known source of
noise rather than as a separate manipulated variable, and we report it as Limitation L4.


In [ ]:
import pandas as pd
import numpy as np
import random
import torch
from skmultilearn.model_selection import iterative_train_test_split
from transformers import pipeline

#Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

dfCovid    = pd.read_csv('data/dataRwaCovid.csv')
dfDiabetes = pd.read_csv('data/dataIHADiabetes.csv')

DiabetesTopics = [
    "Physical",
    "Psychological",
    "No Symptoms",
    "Prognosis",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Testing/Monitoring Devices",
    "Health Data",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Adverse Events",
    "Therapeutic Devices",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Insurance/Billing",
    "Medical Records",
    "Referrals",
    "Transportation",
    "Primary (Pharmaceutical Prevention)",
    "Primary (Non-Pharmaceutical Prevention)",
    "Secondary (Pharmaceutical Prevention)",
    "Secondary (Non-Pharmaceutical Prevention)",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Entertainment",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety Concerns",
    "Health Education",
    "Sexual & Reproductive Health",
    "Child & Family Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "Rapport",
    "Transition to Adult Clinic"
]

CovidTopics = [
    "Physical",
    "Mental/Emotional",
    "No Symptoms",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Pharmaceutical Prevention",
    "Non-Pharmaceutical Prevention",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety concern",
    "Health Education",
    "Maternal & Child Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "wave",
    "batch"
]

commonSubtopics = sorted(list(set(DiabetesTopics) & set(CovidTopics)))

diabetesColumns = commonSubtopics[:]
diabetesColumns.insert(0,"conversation")
dfDiabetesNew = dfDiabetes[diabetesColumns]

covidColumns = commonSubtopics[:]
covidColumns.insert(0,"conversation(english_only)")
dfCovidNew = dfCovid[covidColumns]

dfCovidNew = dfCovidNew.rename(columns={"conversation(english_only)":"conversation"})

dfCombined = pd.concat([dfCovidNew, dfDiabetesNew])


n = 100
label_sum        = dfCombined[commonSubtopics].sum()
label_keep       = label_sum[label_sum >= n].index.tolist()
label_keep_original = label_keep[:]
label_keep.insert(0, "conversation")

dfCombined_filtered = dfCombined[label_keep]
print(f"Combined filtered shape: {dfCombined_filtered.shape}")
print(f"Labels kept ({len(label_keep_original)}): {label_keep_original}")

x = dfCombined_filtered["conversation"].to_numpy()
y = dfCombined_filtered[label_keep_original].to_numpy()

# np.random.seed(42)
# x_train, y_train, x_, y_ = iterative_train_test_split(
#     x.reshape(-1, 1), y, test_size=0.3
# )
split     = np.load('data/shared_split_indices.npz')
train_idx = split['train_idx']
x_train   = x[train_idx]
y_train   = y[train_idx]
print(f"x_train shape before augmentation: {x_train.shape}")

print("Loading translation models...")
trans_en_to_de = pipeline("translation_en_to_de", model="Helsinki-NLP/opus-mt-en-de",
                           device=0, clean_up_tokenization_spaces=True)
trans_de_to_en = pipeline("translation_de_to_en", model="Helsinki-NLP/opus-mt-de-en",
                           device=0, clean_up_tokenization_spaces=True)

def back_translate(texts, batch_size=32):
    de_trans  = trans_en_to_de(texts, batch_size=batch_size, truncation=True)
    de_texts  = [d["translation_text"] for d in de_trans]
    en_trans  = trans_de_to_en(de_texts, batch_size=batch_size, truncation=True)
    aug_texts = [d["translation_text"] for d in en_trans]
    return aug_texts

print("Running backtranslation...")
orig_texts = x_train.flatten().tolist()
aug_texts  = back_translate(orig_texts)
print("Backtranslation done.")

x_train_aug = np.array(orig_texts + aug_texts)
y_train_aug = np.vstack([y_train, y_train])   # labels are identical for augmented copies

df_out = pd.concat([
    pd.DataFrame({'conversation': x_train_aug}),
    pd.DataFrame(y_train_aug, columns=label_keep_original)
], axis=1)

df_out.to_csv('data/backtranslated_train.csv', index=False)
print(f"Saved backtranslated_train.csv — shape: {df_out.shape}")